# Etap 2 — Przygotowanie danych (ArtiFact) — AI Image Detector

**Cel notebooka:** pobrać dataset **ArtiFact** (`bitmind/ArtiFact`) z Hugging Face na zamontowany Google Drive,
zbudować zbalansowany (50/50 Real vs AI) i zróżnicowany generatorowo zbiór, podzielić na **train / val / test**
i przygotować **preprocessing + augmentację** (transforms PyTorch).

**Ważne fakty o ArtiFact (`bitmind/ArtiFact`):**
- ~2,5 mln obrazów (965k real / 1,53M fake), **25 generatorów** (13 GAN, 7 Diffusion, 5 inne).
- Rozdzielczość **200×200 px**, całość **~31,7 GB** — pobieramy całość i próbkujemy lokalnie.
- Obrazy są już przetworzone "real-world style" (random crop, downscale, JPEG) wg standardu IEEE VIP Cup 2022.

**Metodologia (do opisania w pracy):**
1. Balans klas 50/50 — eliminuje bias klasowy.
2. Limit obrazów na generator (*cap*) — wymusza różnorodność, zapobiega dominacji jednego modelu.
3. Stratyfikowany split po `(klasa × generator)` — proporcjonalna reprezentacja w każdym splicie.
4. **Midjourney** (z GenImage) jest *held-out* — pojawi się wyłącznie w osobnym teście generalizacji (osobny notebook 2b).

> Notebook jest iteracyjny. Najpierw uruchom komórkę "discovery" (sekcja 3), zobacz realną strukturę plików,
> i dopiero potem budujemy indeks. Jeśli układ folderów będzie inny niż zakładany — dostosujemy parser razem.


## 1. Konfiguracja

Wszystkie parametry w jednym miejscu — łatwo zmieniać i opisać w pracy. Ustawiamy ziarno losowości
(`SEED`) dla powtarzalności wyników (wymóg pracy dyplomowej: reprodukowalność).


In [ ]:
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Ścieżki (Google Drive zamontowany w sekcji 2) ---
DRIVE_ROOT   = "/content/drive/MyDrive/ai-image-detector"   # główny katalog projektu na Drive
ARTIFACT_DIR = os.path.join(DRIVE_ROOT, "artifact_raw")     # tu trafi pobrany ArtiFact
MANIFEST_DIR = os.path.join(DRIVE_ROOT, "manifests")        # tu zapiszemy CSV ze splitami

# --- Parametry zbioru (zgodne z uzgodnioną tabelą) ---
IMG_SIZE      = 224          # wejście modelu (ViT-base / EfficientNet-B0 = 224)
PER_GEN_CAP   = 20000        # max obrazów na pojedynczy generator (różnorodność)
N_TRAIN_PER_CLASS = 80000    # docelowo ~80k real + ~80k fake w treningu
N_VAL_PER_CLASS   = 10000
N_TEST_PER_CLASS  = 10000

os.makedirs(MANIFEST_DIR, exist_ok=True)
print("Konfiguracja gotowa. Cel:",
      N_TRAIN_PER_CLASS, "train /", N_VAL_PER_CLASS, "val /", N_TEST_PER_CLASS, "test (na klasę)")


## 2. Montowanie Google Drive i instalacja zależności

Montujemy Drive (tu mamy 1 TB, więc miejsca nie zabraknie) i instalujemy `huggingface_hub`
(do pobrania) oraz upewniamy się co do `torch`/`torchvision` (w Colab są domyślnie).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/ai-image-detector", exist_ok=True)


In [ ]:
# huggingface_hub do pobrania snapshotu; reszta jest w Colab domyślnie
!pip install -q -U "huggingface_hub>=0.23" pandas scikit-learn

import torch, torchvision
print("torch:", torch.__version__, "| CUDA dostępna:", torch.cuda.is_available())


## 3. Pobranie ArtiFact z Hugging Face + rozpoznanie struktury

`snapshot_download` ściąga cały dataset (~31,7 GB) do `ARTIFACT_DIR` na Drive.
Pobranie jest **wznawialne** — jeśli Colab się rozłączy, ponowne uruchomienie dociągnie brakujące pliki
(nie pobiera od zera).

> Pierwsze pobranie potrwa kilkadziesiąt minut. Plik trafia na Drive, więc robisz to **raz** —
> w kolejnych sesjach tylko montujesz Drive i pomijasz tę komórkę.


In [ ]:
from huggingface_hub import snapshot_download

# repo_type="dataset" bo to dataset, nie model
local_path = snapshot_download(
    repo_id="bitmind/ArtiFact",
    repo_type="dataset",
    local_dir=ARTIFACT_DIR,
    local_dir_use_symlinks=False,   # realne pliki na Drive (nie symlinki do cache)
)
print("Pobrano do:", local_path)


**Discovery** — zanim napiszemy parser, oglądamy realny układ plików.
ArtiFact zwykle ma jeden folder na źródło/generator, a w każdym `metadata.csv` z kolumnami `image_path,target`
(`target`: 0 = real, 1 = fake). Uruchom poniższe i — jeśli struktura odbiega od założeń — podeślij mi wynik,
dostosujemy parser.


In [ ]:
import glob, os

# Pokaż foldery najwyższego poziomu
top = sorted([d for d in os.listdir(ARTIFACT_DIR) if os.path.isdir(os.path.join(ARTIFACT_DIR, d))])
print("Foldery najwyższego poziomu (%d):" % len(top))
print(top)

# Znajdź wszystkie metadata.csv
metas = glob.glob(os.path.join(ARTIFACT_DIR, "**", "metadata.csv"), recursive=True)
print("\nZnaleziono metadata.csv:", len(metas))
for m in metas[:5]:
    print(" ", os.path.relpath(m, ARTIFACT_DIR))

# Podejrzyj jeden metadata.csv
if metas:
    import pandas as pd
    sample = pd.read_csv(metas[0])
    print("\nKolumny w metadata.csv:", list(sample.columns))
    print(sample.head())


## 4. Budowa jednolitego indeksu

Łączymy wszystkie `metadata.csv` w jeden DataFrame z kolumnami:
- `path` — pełna ścieżka do obrazu,
- `label` — 0 (real) / 1 (fake),
- `generator` — nazwa folderu źródła/generatora (do balansu i stratyfikacji).

Folder nadrzędny pełni rolę etykiety generatora (np. `biggan`, `stable_diffusion`, `ffhq`).


In [ ]:
import glob, os
import pandas as pd

metas = glob.glob(os.path.join(ARTIFACT_DIR, "**", "metadata.csv"), recursive=True)
assert metas, "Nie znaleziono metadata.csv — uruchom komórkę discovery i sprawdź strukturę."

rows = []
for m in metas:
    folder = os.path.dirname(m)
    generator = os.path.basename(folder)          # nazwa źródła/generatora = nazwa folderu
    df = pd.read_csv(m)
    # standaryzacja nazw kolumn (różne wersje ArtiFact bywają: image_path/target)
    cols = {c.lower(): c for c in df.columns}
    path_col   = cols.get("image_path") or cols.get("path") or list(df.columns)[0]
    label_col  = cols.get("target") or cols.get("label")
    for _, r in df.iterrows():
        rows.append({
            "path": os.path.join(folder, str(r[path_col])),
            "label": int(r[label_col]),
            "generator": generator,
        })

index = pd.DataFrame(rows)
print("Łącznie obrazów w indeksie:", len(index))
print("\nRozkład klas (0=real, 1=fake):")
print(index["label"].value_counts())
print("\nObrazów na generator:")
print(index["generator"].value_counts())


## 5. Cap per generator + balans 50/50

Dwa kroki:
1. **Cap** — z każdego generatora bierzemy maks. `PER_GEN_CAP` obrazów (losowo, ze stałym ziarnem).
   Dzięki temu duży generator nie zdominuje zbioru i model nie nauczy się artefaktów jednego modelu.
2. **Balans** — wyrównujemy liczbę real i fake do wspólnego budżetu
   (`N_TRAIN+N_VAL+N_TEST` na klasę), żeby zbiór był dokładnie 50/50.


In [ ]:
def cap_per_generator(df, cap, seed=SEED):
    return (df.groupby("generator", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), cap), random_state=seed)))

capped = cap_per_generator(index, PER_GEN_CAP)

budget = N_TRAIN_PER_CLASS + N_VAL_PER_CLASS + N_TEST_PER_CLASS  # na klasę
real = capped[capped.label == 0]
fake = capped[capped.label == 1]
print(f"Po cap: real={len(real)}, fake={len(fake)}, budżet/klasa={budget}")

# Jeśli budżet > dostępne, bierzemy ile się da (i informujemy)
n = min(budget, len(real), len(fake))
if n < budget:
    print(f"UWAGA: dostępne {n}/klasa < budżet {budget}. Zmniejsz cap-zależne N_* lub zwiększ PER_GEN_CAP.")

real = real.sample(n, random_state=SEED)
fake = fake.sample(n, random_state=SEED)
balanced = pd.concat([real, fake]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Zbalansowany zbiór:", len(balanced), "| rozkład:", balanced.label.value_counts().to_dict())


## 6. Stratyfikowany split train / val / test

Stratyfikujemy po kolumnie pomocniczej `strat = label × generator`, żeby **każdy generator i każda klasa**
były proporcjonalnie reprezentowane w train/val/test. Robimy to w dwóch krokach
(`train_test_split` dzieli na 2, więc najpierw odcinamy test, potem val z reszty).
Zapisujemy manifesty CSV na Drive — trening (Etap 3) czyta gotowe listy, niezależnie od tego notebooka.


In [ ]:
from sklearn.model_selection import train_test_split

balanced["strat"] = balanced["label"].astype(str) + "_" + balanced["generator"].astype(str)

test_frac = N_TEST_PER_CLASS / (N_TRAIN_PER_CLASS + N_VAL_PER_CLASS + N_TEST_PER_CLASS)
val_frac  = N_VAL_PER_CLASS  / (N_TRAIN_PER_CLASS + N_VAL_PER_CLASS)

trainval, test = train_test_split(
    balanced, test_size=test_frac, stratify=balanced["strat"], random_state=SEED)
train, val = train_test_split(
    trainval, test_size=val_frac, stratify=trainval["strat"], random_state=SEED)

for name, part in [("train", train), ("val", val), ("test", test)]:
    out = os.path.join(MANIFEST_DIR, f"{name}.csv")
    part[["path", "label", "generator"]].to_csv(out, index=False)
    print(f"{name}: {len(part):>6}  | real/fake = {part.label.value_counts().to_dict()}  -> {out}")


## 7. Preprocessing i augmentacja

**Normalizacja:** statystyki ImageNet (`mean/std`) — bo model startuje z wag pre-trenowanych na ImageNet,
więc wejście musi mieć ten sam rozkład.

**Augmentacja — uwaga metodologiczna (ważne do pracy):** detekcja AI opiera się często na subtelnych
**artefaktach wysokoczęstotliwościowych** (fingerprint generatora). Zbyt agresywna augmentacja (mocny blur,
silna kompresja) potrafi te artefakty *zatrzeć* i pogorszyć detekcję. Dlatego stosujemy **umiarkowaną**
augmentację:
- `RandomResizedCrop` (lekki zakres skali) + `HorizontalFlip` — niezmienniki naturalne,
- delikatny `ColorJitter` — odporność na korekcję barw,
- *opcjonalnie* lekki `RandomJPEG`/blur z małym prawdopodobieństwem — poprawia odporność na realne uploady
  z weba, ale trzeba zważyć kompromis (artefakty vs robustność). Zostawiam zakomentowane do eksperymentu.

Walidacja/test: bez augmentacji (tylko resize + center crop + normalizacja) — deterministyczna ewaluacja.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),  # lekki crop, nie niszczy artefaktów
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    # transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),  # opcjonalnie: robustność web
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),   # 256 dla 224
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ArtifactDataset(Dataset):
    """Czyta obrazy z manifestu CSV (kolumny: path, label, generator)."""
    def __init__(self, manifest_csv, transform):
        self.df = pd.read_csv(manifest_csv)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")   # convert -> zawsze 3 kanały
        return self.transform(img), int(row["label"])

train_ds = ArtifactDataset(os.path.join(MANIFEST_DIR, "train.csv"), train_tf)
val_ds   = ArtifactDataset(os.path.join(MANIFEST_DIR, "val.csv"),   eval_tf)
test_ds  = ArtifactDataset(os.path.join(MANIFEST_DIR, "test.csv"),  eval_tf)
print("Datasety:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# DataLoadery — num_workers=2 to bezpieczny default w Colab; pin_memory przyspiesza transfer na GPU
BATCH = 64
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("Liczba batchy: train=%d val=%d test=%d" % (len(train_dl), len(val_dl), len(test_dl)))


## 8. Kontrola poprawności (sanity check)

Zanim ruszymy z treningiem: sprawdzamy kształt batcha, balans klas i oglądamy kilka obrazów
(z odwróconą normalizacją). Jeśli obrazy wyglądają sensownie i klasy są ~50/50 — dane są gotowe na Etap 3.


In [ ]:
import matplotlib.pyplot as plt
import torch

xb, yb = next(iter(train_dl))
print("Batch X:", tuple(xb.shape), "| Batch y:", tuple(yb.shape))
print("Rozkład klas w batchu:", torch.bincount(yb).tolist())

# de-normalizacja do podglądu
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std  = torch.tensor(IMAGENET_STD).view(3,1,1)

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, lab in zip(axs.flat, xb, yb):
    ax.imshow((img*std + mean).permute(1,2,0).clamp(0,1).numpy())
    ax.set_title("AI (1)" if lab.item()==1 else "Real (0)")
    ax.axis("off")
plt.tight_layout(); plt.show()


---
## Co dalej

- **Etap 2b (krótki):** osobny notebook pobierze **tylko Midjourney z GenImage**, nałoży to samo
  przetwarzanie (resize 200 + JPEG) i zbuduje manifest `test_heldout_midjourney.csv` — czysty test generalizacji.
- **Etap 3:** ładowanie pre-trenowanego ViT/EfficientNet z Hugging Face, podmiana głowicy, pętla treningowa.

Gdy uruchomisz sekcje 2–8 i zobaczysz sensowne obrazy + balans ~50/50, daj znać — przechodzimy do modelu.
